# ??? NEXUS-9: Notebook 03 ? Causal Discovery & Effect Validation

**Objective:**
1. Load synthetic ground-truth causal DAG from `ground_truth_causal_graph.json`.
2. Perform observational Causal Discovery (PC / correlation skeleton).
3. Evaluate discovery metrics against ground truth (SHD, Precision, Recall, F1).
4. Estimate Average Treatment Effect (ATE) under covariate adjustment.
5. Execute Placebo Treatment Refutation to guard against confounding fallacies.


In [1]:
# Step 1: Import dependencies and causal components
import sys
from pathlib import Path
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from nexus9.data.loader import load_futurebench_data
from nexus9.causal_engine.graph import CausalGraph
from nexus9.causal_engine.discovery import discover_causal_skeleton_correlation
from nexus9.causal_engine.effects import estimate_average_treatment_effect
from nexus9.causal_engine.sensitivity import placebo_treatment_refuter
from futurebench.evaluation.causal_metrics import evaluate_causal_discovery
from nexus9.simulator.runner import run_scenario
from nexus9.simulator.scenario import Scenario

print('Causal engine components loaded.')


Causal engine components loaded.


## 1. Ground Truth Causal Graph Inspection
Load the intended data-generating DAG.


In [2]:
# Load ground truth causal graph
gt_graph = CausalGraph.load_ground_truth(PROJECT_ROOT / 'datasets/metadata/ground_truth_causal_graph.json')
print(f'Is Directed Acyclic Graph (DAG): {gt_graph.is_dag()}')
print(f'Nodes ({len(gt_graph.graph.nodes)}): {list(gt_graph.graph.nodes)}')
print(f'Ground-Truth Edges count: {len(gt_graph.graph.edges)}')

# Inspect direct causes of disruption_risk in ground truth
print('Direct parents (causes) of disruption_risk in ground truth:')
print(gt_graph.get_parents('disruption_risk'))


Is Directed Acyclic Graph (DAG): True
Nodes (14): ['economic_pressure', 'weather_severity', 'supplier_reliability', 'demand_shock', 'policy_strength', 'capacity', 'demand', 'supplier_delay_days', 'available_capacity', 'inventory_gap', 'delivery_delay_days', 'operating_cost', 'service_level', 'disruption_risk']
Ground-Truth Edges count: 22
Direct parents (causes) of disruption_risk in ground truth:
['delivery_delay_days', 'inventory_gap', 'service_level']


## 2. Observational Causal Discovery
Infer causal structure directly from observational data without access to ground truth.


In [3]:
# Run causal discovery algorithm
df = load_futurebench_data()
discovered_graph = discover_causal_skeleton_correlation(df, threshold=0.25)

print(f'Discovered Edges Count: {len(discovered_graph.get_edges())}')
print('Sample Discovered Edges:')
print(list(discovered_graph.get_edges())[:10])


Discovered Edges Count: 50
Sample Discovered Edges:
[('weather_severity', 'delivery_delay_days'), ('supplier_delay_days', 'operating_cost'), ('economic_pressure', 'disruption_risk'), ('available_capacity', 'operating_cost'), ('supplier_reliability', 'service_level'), ('demand_shock', 'operating_cost'), ('capacity', 'operating_cost'), ('inventory_gap', 'operating_cost'), ('economic_pressure', 'inventory_gap'), ('supplier_delay_days', 'service_level')]


## 3. Evaluate Causal Discovery against Ground Truth
Compute Structural Hamming Distance (SHD), Precision, Recall, and Causal F1.


In [4]:
# Compare discovered graph against ground truth DAG
eval_results = evaluate_causal_discovery(
    predicted_edges=discovered_graph.get_edges(),
    ground_truth_edges=gt_graph.get_edges(),
)

print('=== CAUSAL DISCOVERY EVALUATION METRICS ===')
for k, v in eval_results.items():
    if isinstance(v, float):
        print(f'{k:20s}: {v:.4f}')
    else:
        print(f'{k:20s}: {v}')


=== CAUSAL DISCOVERY EVALUATION METRICS ===
true_positives      : 18
false_positives     : 32
false_negatives     : 4
precision           : 0.3600
recall              : 0.8182
f1                  : 0.5000
shd                 : 36


## 4. Average Treatment Effect (ATE) Estimation
Estimate the causal effect of `weather_severity` on `disruption_risk` controlling for confounders (`economic_pressure`, `policy_strength`).


In [5]:
# Estimate Average Treatment Effect using OLS Covariate Adjustment
ate_result = estimate_average_treatment_effect(
    df=df,
    treatment='weather_severity',
    outcome='disruption_risk',
    confounders=['economic_pressure', 'policy_strength'],
)

print('=== AVERAGE TREATMENT EFFECT (ATE) ESTIMATION ===')
for k, v in ate_result.items():
    print(f'{k:15s}: {v}')


=== AVERAGE TREATMENT EFFECT (ATE) ESTIMATION ===
treatment      : weather_severity
outcome        : disruption_risk
ate            : 0.5719596717711645
p_value        : 9.4014245009532e-114
ci_lower       : 0.5225044598157084
ci_upper       : 0.6214148837266206


## 5. Placebo Treatment Refutation Test
Randomize the treatment vector to verify that the estimated treatment effect drops to ~0.


In [6]:
# Run Placebo Refutation Test (20 permutations)
refutation_res = placebo_treatment_refuter(
    df=df,
    treatment='weather_severity',
    outcome='disruption_risk',
    confounders=['economic_pressure', 'policy_strength'],
    n_permutations=20,
)

print('=== PLACEBO REFUTATION TEST ===')
for k, v in refutation_res.items():
    print(f'{k:20s}: {v}')

if refutation_res['passed_refutation']:
    print('SUCCESS: Causal estimate passed placebo refutation check.')
else:
    print('CAUTION: Residual effect detected under randomized placebo.')


=== PLACEBO REFUTATION TEST ===
placebo_ate_mean    : 0.008022916280239966
placebo_ate_std     : 0.027191399257534117
passed_refutation   : True
SUCCESS: Causal estimate passed placebo refutation check.


## 6. Simulated Intervention vs. Passive Observation
Observe how the Future Simulator enacts $do(\text{weather\_severity} = 0.95)$ vs baseline.


In [7]:
# Compare baseline vs intervention scenario in simulator
baseline_scen = Scenario(name='Baseline', regime='normal', duration_steps=10)
shock_scen = Scenario(
    name='Severe Weather Intervention',
    regime='extreme_weather',
    interventions={'weather_severity': 0.95},
    duration_steps=10
)

df_base = run_scenario(baseline_scen, n_runs=15)
df_shock = run_scenario(shock_scen, n_runs=15)

print(f'Baseline Mean Disruption Risk:      {df_base["disruption_risk"].mean():.2%}')
print(f'Intervention Mean Disruption Risk:  {df_shock["disruption_risk"].mean():.2%}')
print(f'Baseline Mean Service Level:        {df_base["service_level"].mean():.1f}%')
print(f'Intervention Mean Service Level:    {df_shock["service_level"].mean():.1f}%')


Baseline Mean Disruption Risk:      0.00%
Intervention Mean Disruption Risk:  85.33%
Baseline Mean Service Level:        98.8%
Intervention Mean Service Level:    79.2%
